# Segment 3: Embeddings

## Claude API for Python Developers

In this segment, we'll cover:
- Understanding Embeddings and Vector Representations
- Question Answering with Semantic Search
- Building Recommendation Systems
- Handling Long Texts

**Note**: Anthropic recommends Voyage AI for embeddings. We'll use the `voyageai` library.


## Setup and Imports


In [ ]:
import voyageai
import anthropic
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

# Initialize clients
# Both use environment variables: VOYAGE_API_KEY and ANTHROPIC_API_KEY
vo = voyageai.Client()
claude = anthropic.Anthropic()

print("✅ Voyage AI client initialized")
print("✅ Anthropic client initialized")


---
## 1. Understanding Embeddings

### What are Embeddings?

Embeddings are **dense vector representations** of text that capture semantic meaning. Similar texts have similar vectors, allowing us to:

- Find semantically similar documents
- Build search systems that understand meaning
- Create recommendation engines
- Cluster related content

### Why Voyage AI?

Anthropic recommends Voyage AI as their embedding partner because:
- Optimized for use with Claude
- High-quality embeddings for retrieval
- Multiple model options for different use cases


In [ ]:
# Generate your first embedding
text = "Machine learning is a subset of artificial intelligence."

result = vo.embed(
    texts=[text],
    model="voyage-2",  # General-purpose embedding model
    input_type="document"
)

embedding = result.embeddings[0]

print(f"Text: '{text}'")
print(f"\nEmbedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"\nEmbedding type: {type(embedding)}")


In [ ]:
# Visualizing semantic similarity
# Similar texts should have similar embeddings (high cosine similarity)

texts = [
    "The cat sat on the mat.",
    "A kitten was resting on the rug.",
    "Python is a programming language.",
    "Dogs are loyal pets.",
    "Machine learning models process data."
]

# Generate embeddings for all texts
result = vo.embed(
    texts=texts,
    model="voyage-2",
    input_type="document"
)

embeddings = np.array(result.embeddings)

# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(embeddings)

# Display results
print("Texts:")
for i, t in enumerate(texts):
    print(f"  {i}: {t}")

print("\nCosine Similarity Matrix:")
print("     ", end="")
for i in range(len(texts)):
    print(f"  {i}   ", end="")
print()

for i in range(len(texts)):
    print(f"  {i}: ", end="")
    for j in range(len(texts)):
        print(f"{similarity_matrix[i][j]:.3f} ", end="")
    print()


In [ ]:
# Interpreting similarity scores
print("🔍 Similarity Insights:\n")

pairs = [
    (0, 1, "Cat on mat vs Kitten on rug (similar meaning)"),
    (0, 2, "Cat on mat vs Python programming (different topics)"),
    (0, 3, "Cat on mat vs Dogs are pets (both about animals)"),
    (2, 4, "Python programming vs ML models (both tech)")
]

for i, j, description in pairs:
    sim = similarity_matrix[i][j]
    print(f"{description}")
    print(f"  Similarity: {sim:.3f}")
    print()


---
## 2. Question Answering with Semantic Search

Build a simple Q&A system that:
1. Creates embeddings for a knowledge base
2. Finds relevant passages using semantic search
3. Uses Claude to generate answers based on retrieved context


In [ ]:
# Sample knowledge base about Python
knowledge_base = [
    {
        "id": 1,
        "title": "Python Basics",
        "content": "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasizes code readability with its notable use of significant indentation."
    },
    {
        "id": 2,
        "title": "Python Data Types",
        "content": "Python has several built-in data types including integers, floats, strings, lists, tuples, dictionaries, and sets. Variables in Python are dynamically typed, meaning you don't need to declare their type."
    },
    {
        "id": 3,
        "title": "Python Functions",
        "content": "Functions in Python are defined using the 'def' keyword. They can accept parameters, return values, and support default arguments, keyword arguments, and variable-length arguments with *args and **kwargs."
    },
    {
        "id": 4,
        "title": "Python Classes",
        "content": "Python supports object-oriented programming with classes. Classes are defined using the 'class' keyword and can have methods, attributes, inheritance, and special methods like __init__ and __str__."
    },
    {
        "id": 5,
        "title": "Python List Comprehensions",
        "content": "List comprehensions provide a concise way to create lists. The syntax is [expression for item in iterable if condition]. They are more readable and often faster than traditional loops."
    },
    {
        "id": 6,
        "title": "Python Virtual Environments",
        "content": "Virtual environments isolate Python dependencies for different projects. Tools like venv, virtualenv, and conda help manage these environments. They prevent conflicts between package versions."
    },
    {
        "id": 7,
        "title": "Python Package Management",
        "content": "pip is Python's package installer. It installs packages from PyPI (Python Package Index). requirements.txt files list project dependencies. Modern alternatives include Poetry and uv."
    },
    {
        "id": 8,
        "title": "Python Error Handling",
        "content": "Python uses try/except blocks for error handling. Common exceptions include ValueError, TypeError, KeyError, and FileNotFoundError. You can also create custom exceptions by inheriting from Exception."
    }
]

print(f"Knowledge base contains {len(knowledge_base)} documents")


In [ ]:
# Create embeddings for the knowledge base
doc_texts = [doc["content"] for doc in knowledge_base]

doc_result = vo.embed(
    texts=doc_texts,
    model="voyage-2",
    input_type="document"
)

doc_embeddings = np.array(doc_result.embeddings)

print(f"Created embeddings for {len(doc_embeddings)} documents")
print(f"Embedding shape: {doc_embeddings.shape}")


In [ ]:
def semantic_search(query, doc_embeddings, knowledge_base, top_k=3):
    """
    Find the most relevant documents for a query using semantic search.
    """
    # Generate embedding for the query
    query_result = vo.embed(
        texts=[query],
        model="voyage-2",
        input_type="query"  # Note: use "query" for search queries
    )
    query_embedding = np.array(query_result.embeddings[0])
    
    # Calculate cosine similarity with all documents
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    
    # Get top-k most similar documents
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "document": knowledge_base[idx],
            "similarity": similarities[idx]
        })
    
    return results

# Test the search
query = "How do I handle errors in Python?"
results = semantic_search(query, doc_embeddings, knowledge_base)

print(f"Query: '{query}'")
print("\n📚 Top relevant documents:\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['document']['title']} (similarity: {r['similarity']:.3f})")
    print(f"   {r['document']['content'][:100]}...")
    print()


In [ ]:
def answer_question(query, doc_embeddings, knowledge_base, claude_client, top_k=3):
    """
    Answer a question using RAG (Retrieval-Augmented Generation):
    1. Find relevant documents via semantic search
    2. Use Claude to generate an answer based on the context
    """
    # Step 1: Retrieve relevant documents
    results = semantic_search(query, doc_embeddings, knowledge_base, top_k)
    
    # Step 2: Build context from retrieved documents
    context = "\n\n".join([
        f"Document: {r['document']['title']}\n{r['document']['content']}"
        for r in results
    ])
    
    # Step 3: Generate answer with Claude
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        system="""You are a helpful Python tutor. Answer questions based on the provided context.
If the context doesn't contain enough information, say so.
Be concise and provide code examples when helpful.""",
        messages=[{
            "role": "user",
            "content": f"""
<context>
{context}
</context>

<question>
{query}
</question>

Please answer the question based on the context provided.
"""
        }]
    )
    
    return {
        "answer": response.content[0].text,
        "sources": [r["document"]["title"] for r in results]
    }

# Test the Q&A system
query = "How do I create a function with default parameters?"
result = answer_question(query, doc_embeddings, knowledge_base, claude)

print(f"❓ Question: {query}")
print(f"\n📖 Sources: {', '.join(result['sources'])}")
print(f"\n💡 Answer:\n{result['answer']}")


In [ ]:
# Try more questions
questions = [
    "What are the different data types in Python?",
    "How do I manage project dependencies?",
    "What is the difference between a list and a tuple?"
]

for q in questions:
    result = answer_question(q, doc_embeddings, knowledge_base, claude)
    print(f"❓ {q}")
    print(f"📖 Sources: {', '.join(result['sources'])}")
    print(f"💡 {result['answer'][:200]}...")
    print("\n" + "="*60 + "\n")


---
## 3. Recommendations

Embeddings can power recommendation systems by finding similar items.


In [ ]:
# Sample product catalog
products = [
    {"id": 1, "name": "Laptop Stand", "description": "Ergonomic aluminum laptop stand with adjustable height for better posture while working."},
    {"id": 2, "name": "Mechanical Keyboard", "description": "RGB mechanical keyboard with Cherry MX switches for comfortable typing and gaming."},
    {"id": 3, "name": "Wireless Mouse", "description": "Ergonomic wireless mouse with precision tracking and programmable buttons."},
    {"id": 4, "name": "Monitor Arm", "description": "Adjustable monitor arm that clamps to desk, frees up space and improves viewing angle."},
    {"id": 5, "name": "USB Hub", "description": "7-port USB 3.0 hub with fast data transfer for connecting multiple devices."},
    {"id": 6, "name": "Webcam HD", "description": "1080p HD webcam with built-in microphone for video calls and streaming."},
    {"id": 7, "name": "Desk Mat", "description": "Large desk mat with smooth surface for mouse tracking and wrist comfort."},
    {"id": 8, "name": "Cable Management Kit", "description": "Complete cable organizer set with clips, ties, and sleeves to keep desk tidy."},
    {"id": 9, "name": "Ring Light", "description": "LED ring light with adjustable brightness for video calls and content creation."},
    {"id": 10, "name": "Noise-Canceling Headphones", "description": "Wireless headphones with active noise cancellation for focused work and music."}
]

# Create embeddings for products
product_texts = [f"{p['name']}: {p['description']}" for p in products]
product_result = vo.embed(texts=product_texts, model="voyage-2", input_type="document")
product_embeddings = np.array(product_result.embeddings)

print(f"Created embeddings for {len(products)} products")


In [ ]:
def get_recommendations(product_id, product_embeddings, products, top_k=3):
    """
    Get product recommendations based on similarity to a given product.
    """
    # Get the embedding for the target product
    product_idx = product_id - 1  # Convert to 0-based index
    target_embedding = product_embeddings[product_idx]
    
    # Calculate similarity with all products
    similarities = cosine_similarity([target_embedding], product_embeddings)[0]
    
    # Get top-k similar products (excluding the target itself)
    top_indices = np.argsort(similarities)[::-1]
    
    recommendations = []
    for idx in top_indices:
        if idx != product_idx and len(recommendations) < top_k:
            recommendations.append({
                "product": products[idx],
                "similarity": similarities[idx]
            })
    
    return recommendations

# Demo: Get recommendations for Laptop Stand
product_id = 1  # Laptop Stand
recommendations = get_recommendations(product_id, product_embeddings, products)

print(f"🛍️ Recommendations for: {products[product_id-1]['name']}")
print(f"   {products[product_id-1]['description']}")
print("\n📦 Similar products you might like:\n")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['product']['name']} (similarity: {rec['similarity']:.3f})")
    print(f"   {rec['product']['description']}")
    print()


In [ ]:
# Try recommendations for different products
test_products = [2, 6, 10]  # Mechanical Keyboard, Webcam, Headphones

for pid in test_products:
    recommendations = get_recommendations(pid, product_embeddings, products)
    print(f"🛍️ If you like '{products[pid-1]['name']}', you might also like:")
    for rec in recommendations:
        print(f"   • {rec['product']['name']} ({rec['similarity']:.3f})")
    print()


In [ ]:
# Search-based recommendations
def search_products(query, product_embeddings, products, top_k=3):
    """Find products matching a user's search query."""
    query_result = vo.embed(texts=[query], model="voyage-2", input_type="query")
    query_embedding = np.array(query_result.embeddings[0])
    
    similarities = cosine_similarity([query_embedding], product_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "product": products[idx],
            "similarity": similarities[idx]
        })
    return results

# Test search queries
queries = [
    "I need something for video calls",
    "help me organize my desk",
    "comfortable typing experience"
]

for query in queries:
    results = search_products(query, product_embeddings, products)
    print(f"🔍 Search: '{query}'")
    for r in results:
        print(f"   • {r['product']['name']} ({r['similarity']:.3f})")
    print()


---
## 4. Handling Long Texts

When documents are too long, we need strategies for:
1. Chunking documents into smaller pieces
2. Creating embeddings for chunks
3. Retrieving and reassembling relevant chunks


In [ ]:
# Sample long document
long_document = """
# Complete Guide to Python Web Development

## Chapter 1: Introduction to Web Frameworks

Python offers several web frameworks for building applications. The most popular ones are 
Django and Flask. Django is a full-featured framework that includes an ORM, admin interface, 
and authentication system. Flask is a micro-framework that gives developers more control 
over component choices.

## Chapter 2: Setting Up Your Development Environment

Before starting web development, you need to set up your environment. Install Python 3.8+, 
create a virtual environment using venv or conda, and install your chosen framework. 
For Django: pip install django. For Flask: pip install flask.

## Chapter 3: Understanding HTTP and REST

Web applications communicate using HTTP protocol. REST (Representational State Transfer) 
is an architectural style for designing networked applications. RESTful APIs use HTTP 
methods: GET for reading, POST for creating, PUT for updating, DELETE for removing.

## Chapter 4: Database Integration

Most web applications need a database. Django includes built-in support for PostgreSQL, 
MySQL, SQLite, and Oracle through its ORM. Flask typically uses SQLAlchemy for database 
operations. Understanding SQL and database design is essential for web developers.

## Chapter 5: Authentication and Security

Security is crucial in web development. Implement user authentication with sessions or 
tokens (JWT). Protect against common vulnerabilities: SQL injection, XSS (Cross-Site 
Scripting), CSRF (Cross-Site Request Forgery). Always use HTTPS in production.

## Chapter 6: Deployment

Deploying Python web apps involves choosing a hosting platform (AWS, Heroku, DigitalOcean), 
setting up a production server (Gunicorn, uWSGI), configuring a reverse proxy (Nginx), 
and implementing CI/CD pipelines for automated deployment.
"""

print(f"Document length: {len(long_document)} characters")
print(f"Approximate words: {len(long_document.split())}")


In [ ]:
def chunk_document(text, chunk_size=500, overlap=50):
    """
    Split document into overlapping chunks.
    
    Args:
        text: The document to chunk
        chunk_size: Target size of each chunk in characters
        overlap: Number of characters to overlap between chunks
    
    Returns:
        List of (chunk_text, start_position, end_position)
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        # Try to end at a sentence boundary
        if end < len(text):
            # Look for sentence endings near the chunk boundary
            for boundary in ['. ', '.\n', '! ', '? ']:
                boundary_pos = text.rfind(boundary, start + chunk_size - 100, end + 50)
                if boundary_pos != -1:
                    end = boundary_pos + len(boundary)
                    break
        
        chunk = text[start:end].strip()
        if chunk:  # Only add non-empty chunks
            chunks.append({
                "text": chunk,
                "start": start,
                "end": end
            })
        
        start = end - overlap
    
    return chunks

# Chunk the document
chunks = chunk_document(long_document, chunk_size=600, overlap=50)

print(f"Document split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk['text'])} chars")
    print(f"  Preview: {chunk['text'][:80]}...")
    print()


In [ ]:
# Create embeddings for chunks
chunk_texts = [c["text"] for c in chunks]
chunk_result = vo.embed(texts=chunk_texts, model="voyage-2", input_type="document")
chunk_embeddings = np.array(chunk_result.embeddings)

def search_chunks(query, chunk_embeddings, chunks, top_k=2):
    """Search for relevant chunks in the document."""
    query_result = vo.embed(texts=[query], model="voyage-2", input_type="query")
    query_embedding = np.array(query_result.embeddings[0])
    
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "chunk_id": idx,
            "similarity": similarities[idx]
        })
    return results

# Test chunk search
query = "How do I secure my web application?"
results = search_chunks(query, chunk_embeddings, chunks)

print(f"🔍 Query: '{query}'")
print("\n📄 Relevant sections:\n")
for r in results:
    print(f"Chunk {r['chunk_id']+1} (similarity: {r['similarity']:.3f}):")
    print(f"{r['chunk']['text']}")
    print("\n" + "-"*50 + "\n")


In [ ]:
# Complete RAG pipeline for long documents
def answer_from_document(query, chunk_embeddings, chunks, claude_client, top_k=2):
    """
    Answer questions about a long document using RAG.
    """
    # Retrieve relevant chunks
    results = search_chunks(query, chunk_embeddings, chunks, top_k)
    
    # Build context from chunks
    context = "\n\n---\n\n".join([r["chunk"]["text"] for r in results])
    
    # Generate answer with Claude
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=400,
        system="You are a helpful assistant answering questions about a document. Use only the provided context.",
        messages=[{
            "role": "user",
            "content": f"""
<context>
{context}
</context>

<question>
{query}
</question>

Answer based on the context provided. If the context doesn't contain the answer, say so.
"""
        }]
    )
    
    return response.content[0].text

# Test with different questions
questions = [
    "What's the difference between Django and Flask?",
    "How do I deploy a Python web application?",
    "What security threats should I protect against?"
]

for q in questions:
    answer = answer_from_document(q, chunk_embeddings, chunks, claude)
    print(f"❓ {q}")
    print(f"💡 {answer}")
    print("\n" + "="*60 + "\n")


---
## Summary

In this segment, we covered:

1. **Understanding Embeddings**: Vector representations that capture semantic meaning
2. **Question Answering**: Building RAG systems with semantic search + Claude
3. **Recommendations**: Finding similar items using embedding similarity
4. **Long Texts**: Chunking strategies for documents exceeding context limits

### Key Takeaways:
- Embeddings enable semantic search beyond keyword matching
- Voyage AI is Anthropic's recommended embedding provider
- Use `input_type="document"` for content and `"query"` for searches
- Combine embeddings with Claude for powerful Q&A systems
- Chunk long documents with overlap to maintain context

### Best Practices:
- Choose chunk size based on your content type
- Use overlap to avoid cutting important context
- Store embeddings for large collections (databases like Pinecone, Weaviate)
- Consider relevance thresholds to filter poor matches

### Next Steps
In Segment 4, we'll explore Claude's tool use capabilities for:
- Structured outputs
- Function calling
- Building agentic workflows
